[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Agregación y tablas de resumen.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo II: *Describir y comparar*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se resumirán datos por grupos con `groupby`, `pivot_table` y `crosstab`, y se cambiará entre formato ancho y largo con `melt`.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

La API que se usa hoy está explicada método por método en py311; aquí solo se repasa y el tiempo se dedica a **interpretar** y a **decidir** qué tabla, qué medida de asociación y qué gráfica responden mejor la pregunta:

| Tema | *Notebooks* de py311 |
|---|---|
| `melt`, `stack`/`unstack`, `pivot_table`, `crosstab` | [21 Transformación de *dataframes* y series](https://github.com/PythonistaMX/py311/blob/main/21_transformacion_de_objetos.ipynb) |
| `groupby`, agregación y `transform` por grupo | [24 Métodos `groupby()`](https://github.com/PythonistaMX/py311/blob/main/24_metodos_groupby.ipynb) |
| *Seaborn*: funciones por tipo de gráfica y la interfaz de objetos | [31 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/31_introduccion_a_seaborn.ipynb) · [32 Objetos de *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/32_objetos_de_seaborn.ipynb) |

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3, así que pueden verse diferencias (por ejemplo, texto como `object` en lugar de `str`, o el uso de `inplace=True`). Cuando difieran, se sigue lo que muestra este *notebook* (ver la sección 1.1 de `03_pandas_3_esencial.ipynb`).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluido el módulo `stats` de *SciPy*, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats

warnings.filterwarnings("ignore", message="vert: bool was deprecated")

**Nota:** el filtro de advertencias oculta solo un aviso interno de *Seaborn* 0.13 con *Matplotlib* 3.11 al dibujar diagramas de caja (`vert: bool was deprecated`). No afecta los resultados.

* La siguiente celda leerá `penguins_limpio.csv` o lo regenerará desde `penguins.csv` si no existe.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio).astype({"species": "category", "island": "category", "sex": "category"})
pinguinos.shape

`tips` registra 244 cuentas de un restaurante: `total_bill` (cuenta, en dólares), `tip` (propina), `sex` y `smoker` de quien paga, `day`, `time` (comida o cena) y `size` (personas en la mesa). Se creará la **propina porcentual** `tip_pct = 100 · tip / total_bill` y se ordenarán los días.

* La siguiente celda leerá `tips.csv`, creará `tip_pct` y convertirá `day` y `time` en categorías ordenadas.

In [ ]:
propinas = (
    pd.read_csv(os.path.join(DATA, "tips.csv"))
    .assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
    .astype({"day": pd.CategoricalDtype(["Thur", "Fri", "Sat", "Sun"], ordered=True),
             "time": pd.CategoricalDtype(["Lunch", "Dinner"], ordered=True)})
)
propinas.head()

---
## 1. Repaso (10 min).

En los *notebooks* `10` a `12` se vio que la masa y la aleta de los pingüinos se describen mal en conjunto y bien **por especie**. Hoy se formaliza esa idea: resumir por grupos y estudiar cómo se relacionan las variables. Se usarán `penguins_limpio` (se regenera si no existe) y `tips`.

---
## 2. Agregación (25 min).

| Herramienta | Para qué | Resultado |
|---|---|---|
| `groupby(...).agg(...)` | Resumir una o varias columnas por grupo | Una fila por grupo |
| `pivot_table` | Resumen con un grupo en filas y otro en columnas | Tabla de doble entrada |
| `crosstab` | Conteos (o proporciones) de dos categóricas | Tabla de contingencia |
| `melt` | Pasar de formato **ancho** (una columna por variable) a **largo** (una fila por observación y variable) | Ideal para *Seaborn* y facetas |

El detalle de cada método está en [py311 · 21](https://github.com/PythonistaMX/py311/blob/main/21_transformacion_de_objetos.ipynb) y [py311 · 24](https://github.com/PythonistaMX/py311/blob/main/24_metodos_groupby.ipynb). Las decisiones que sí hay que tomar al agregar:

* **¿Contar con o sin faltantes?** `count` cuenta valores no nulos y `size` cuenta filas: reportar el *n* de cada grupo junto a cada media.
* **¿Media o mediana por grupo?** La misma regla de `10_tendencia_central_y_dispersion`: depende de la forma de la distribución dentro de cada grupo.
* **¿Proporción respecto a qué?** En `crosstab`, `normalize="index"` y `normalize="columns"` responden preguntas distintas (ejercicio 2).
* **¿Grupos con pocos datos?** Una media calculada con 1 a 5 observaciones no es comparable con una de 80; se reporta el *n* o se agrupan categorías. Por ejemplo, en `tips` la cena del jueves tiene una sola cuenta.

**Nota:** en *pandas* 3, `groupby` sobre columnas categóricas usa `observed=True` por defecto: solo aparecen las combinaciones que existen en los datos.

**Concepto: dividir, aplicar, combinar.** `groupby` **divide** las filas según el valor de la columna de grupo, **aplica** una función a cada parte y **combina** los resultados en una tabla con una fila por grupo. En el flujo del EDA, agregar es el paso que convierte miles de filas en una comparación legible, y cada media va con su *n* para saber cuánto confiar en ella.

* La siguiente celda calculará varias estadísticas de la masa por especie con **agregaciones con nombre**.

In [ ]:
pinguinos.groupby("species")["body_mass_g"].agg(
    n="count",
    media="mean",
    mediana="median",
    desv_est="std",
).round(1)

**Interpretación:** Adelie y Chinstrap tienen casi la misma masa típica (≈ 3,700 g), mientras que los Gentoo pesan unos 1,350 g más (1,343 g más que los Chinstrap y 1,375 g más que los Adelie). En las tres especies media y mediana están cerca, así que la media es una buena descripción del centro. Los *n* son distintos (68 a 151), algo que hay que tener en cuenta al comparar.

> 💡 **El *n* de cada grupo importa:** una media por grupo siempre debe ir con el número de datos que la sostienen. Con 5 observaciones, un dato raro la mueve mucho; con 150, casi nada. Antes de comparar dos grupos, mira cuántos hay en cada uno.

* La siguiente celda usará `pivot_table` para la aleta media por especie (filas) y sexo (columnas), con totales.

In [ ]:
pinguinos.pivot_table(index="species", columns="sex", values="flipper_length_mm",
                      aggfunc="mean", margins=True, margins_name="Total").round(1)

**Interpretación:** en las tres especies los machos tienen la aleta más larga (de 4.6 a 8.8 mm más), pero la diferencia entre especies es mucho mayor (de 190 a 217 mm). Los márgenes (`Total`) se calculan con todas las filas de esa fila o columna, no promediando las celdas: coinciden con el promedio de las celdas solo cuando los grupos tienen el mismo tamaño (como en Adelie, con 73 machos y 73 hembras).

* La siguiente celda usará `crosstab` para contar pingüinos por isla y especie.

In [ ]:
pd.crosstab(pinguinos["island"], pinguinos["species"], margins=True, margins_name="Total")

**Interpretación:** la tabla tiene **ceros estructurales**: los Gentoo solo están en Biscoe y los Chinstrap solo en Dream; Torgersen solo tiene Adelie. Isla y especie están confundidas, así que cualquier comparación entre islas es, en buena parte, una comparación entre especies (el ejemplo de la isla Biscoe en `05_muestreo_y_sesgos_de_recoleccion`).

**Concepto: leer una tabla de conteos por renglón o por columna.** Una tabla de conteos se puede leer de dos maneras. Dividir cada **renglón** entre su total responde «de los pingüinos de Biscoe, ¿qué fracción es de cada especie?» (73.7% son Gentoo y 26.3% son Adelie). Dividir cada **columna** responde «de los Gentoo, ¿qué fracción vive en cada isla?» (el 100% vive en Biscoe). Son preguntas distintas con números distintos: decide cuál es la tuya antes de calcular.

> 🔗 **Puente con el siguiente *notebook*:** cuando dos variables están así de entrelazadas, comparar islas es comparar especies. Esa tercera variable que lo enreda todo se llama **confusor**, y `14_relaciones_entre_variables` muestra cómo se maneja.

* La siguiente celda pasará las cuatro medidas a formato largo con `melt`.

In [ ]:
largo = pinguinos.melt(id_vars=["species", "sex"], value_vars=medidas,
                       var_name="medida", value_name="valor")
largo.head(), largo.shape

**Interpretación:** de 342 filas × 4 medidas se pasa a 1,368 filas (342 × 4) con dos columnas nuevas: `medida` (qué se midió) y `valor`. Es el formato que *Seaborn* necesita para facetar por medida con `col="medida"`.

### Ejercicio 1 · Masa por especie y sexo.

**🎯 Objetivo:** agregar por dos variables y comparar dentro de cada grupo. **Qué se espera aprender:** a construir la tabla que responde una pregunta concreta (¿cuánto más pesan los machos en cada especie?) con su *n*.

Con agregaciones con nombre, calcula para `body_mass_g` por **especie y sexo**: `n`, `media`, `desv_est` y `mediana`. ¿En qué especie es mayor la diferencia entre machos y hembras? (pista: `unstack("sex")` y una resta).

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 2 · Proporciones con `crosstab`.

**🎯 Objetivo:** distinguir proporciones por fila y por columna. **Qué se espera aprender:** que cada forma de calcular la proporción (por fila o por columna) responde una pregunta distinta y que hay que elegirla antes de calcular.

1. Calcula la proporción de cada especie **dentro de cada isla** (`normalize="index"`) y la proporción de cada isla **dentro de cada especie** (`normalize="columns"`). ¿Qué pregunta responde cada tabla?
2. ¿Por qué no se puede comparar "la masa media por isla" sin pensar en la especie?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 3 · Formato largo y facetas.

**🎯 Objetivo:** usar el formato largo para graficar varias medidas a la vez. **Qué se espera aprender:** cuándo conviene `melt` y cómo las facetas evitan amontonar grupos.

Con `largo`, dibuja con `sns.catplot` un diagrama de caja de `valor` por especie, con una faceta por `medida` (`col="medida"`, `sharey=False`, `col_wrap=2`). ¿Qué medida separa mejor a los Gentoo de las otras dos especies?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Supervivencia en el Titanic por clase y sexo.

**🎯 Objetivo:** combinar agregación y proporciones con dos factores. **Qué se espera aprender:** a leer una tabla de tasas por grupo con sus márgenes.

Con `titanic.csv`, construye con `pivot_table` la tasa de supervivencia por `class` (filas, en orden First, Second, Third) y `sex` (columnas), con márgenes. Después obtén la misma tabla con `crosstab(..., values=..., aggfunc="mean")`. ¿Qué variable parece pesar más en la supervivencia?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 2:**
* `groupby().agg()` con nombres, `pivot_table`, `crosstab` y `melt` cubren casi todas las tablas de un EDA.
* Cada media por grupo va con su *n*; las proporciones se calculan por fila o por columna según la pregunta.
* Las tablas de contingencia revelan grupos confundidos (isla y especie).

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios pendientes de este *notebook* y, si es posible, un reto.

**Siguiente *notebook*:** `14_relaciones_entre_variables.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>